# 🎯 Module 03: Modern Task Orchestration & TaskGroups (Python 3.11+)

In Module 02, we learned how to spawn individual background tasks with `asyncio.create_task()`.
Now we explore how to **coordinate multiple concurrent tasks reliably in production**.

### What We Will Master:
1. **`asyncio.gather` Deep Dive**: When to use it, and why it leaks orphaned tasks on failure.
2. **Structured Concurrency with `asyncio.TaskGroup` (Python 3.11+)**: Automatic cancellation cascading.
3. **ExceptionGroups & `except*` Syntax**: Handling multiple simultaneous errors.
4. **Timeouts Done Right**: `asyncio.timeout` vs `asyncio.timeout_at` vs legacy `wait_for`.
5. **Shielding Critical Operations**: `asyncio.shield()`.
6. **Racing Concurrent Tasks**: `asyncio.wait` with `FIRST_COMPLETED` (e.g. multi-LLM race).
7. **🎯 Enterprise Challenge**: Resilient Multi-Provider Price/Stock Harvester.

---
## 1. The Hidden Danger of `asyncio.gather` (Orphaned Tasks)

Consider 3 tasks running in `asyncio.gather(t1, t2, t3)`:
- If `t2` crashes with an exception, `gather` raises that exception to you immediately.
- **CRITICAL FLAW**: `t1` and `t3` **keep running in the background as orphans**! They consume CPU/sockets and may write corrupted partial data.
- If you pass `return_exceptions=True`, it catches the errors, but you must manually inspect every result.

In [1]:
import asyncio
import time

async def task_worker(name: str, delay: float, fail: bool = False):
    print(f"{name} started...")
    await asyncio.sleep(delay)
    if fail:
        raise ConnectionError("DB connection lost!")
    return f"{name}_done"

# Test 1: Default gather behavior
print("--- Test 1: gather() with return_exceptions=False ---")
t_a = asyncio.create_task(task_worker("Task A", 1.0))
t_b = asyncio.create_task(task_worker("Task B", 0.2, fail=True))
t_c = asyncio.create_task(task_worker("Task C", 1.0))

try:
    await asyncio.gather(t_a, t_b, t_c)
except ConnectionError as err:
    print(f"❌ gather() raised: {repr(err)}")
    print("Notice: Task A and C are STILL RUNNING as orphans in the background!")
    print(f"Task A still running? {not t_a.done()}")
    print(f"Task C still running? {not t_c.done()}")

# Clean up
t_a.cancel(); t_c.cancel()
await asyncio.gather(t_a, t_c, return_exceptions=True)

# Test 2: return_exceptions=True
print("\n--- Test 2: gather() with return_exceptions=True ---")
res = await asyncio.gather(
    task_worker("A", 0.1),
    task_worker("B", 0.1, fail=True),
    task_worker("C", 0.1),
    return_exceptions=True
)
print("Results:", res)
print("Notice: No exception was raised; errors are returned as items in the list.")

--- Test 1: gather() with return_exceptions=False ---
Task A started...
Task B started (will crash)...
Task C started...
❌ gather() raised: ConnectionError('DB connection lost!')
Notice: Task A and C are STILL RUNNING as orphans in the background!
Task A still running? True
Task C still running? True

--- Test 2: gather() with return_exceptions=True ---
Results: ['A_done', ConnectionError('DB connection lost!'), 'C_done']
Notice: No exception was raised; errors are returned as items in the list.


---
## 2. Structured Concurrency: `asyncio.TaskGroup` (Python 3.11+)

**Structured Concurrency** means:
> *"If a concurrent operation creates 5 child tasks, all 5 must either finish together, or if ANY child crashes, all remaining siblings are automatically cancelled before exiting the block."*

No more orphaned tasks! This is the modern, battle-tested standard in Python 3.11+.

In [ ]:
async def structured_worker(name: str, delay: float):
    try:
        print(f"[{time.strftime('%X')}] 🟢 {name} started (delay {delay}s)...")
        await asyncio.sleep(delay)
        return f"{name}_done"
    except asyncio.CancelledError:
        print(f"[{time.strftime('%X')}] 🧹 {name} received CancelledError! Cleaned up socket.")
        raise

async def crashing_worker(name: str, delay: float):
    print(f"[{time.strftime('%X')}] 💣 {name} started (will crash at {delay}s)...")
    await asyncio.sleep(delay)
    print(f"[{time.strftime('%X')}] 💥 {name} crashed!")
    raise ValueError(f"{name} encountered bad state!")

try:
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(structured_worker("Worker 1", 2.0))
        t2 = tg.create_task(crashing_worker("Worker 2", 0.3))
        t3 = tg.create_task(structured_worker("Worker 3", 2.0))
except* ValueError as _eg:
    print("🛑 Caught ExceptionGroup! All siblings were safely cancelled:")
    print(f"  Worker 1 cancelled? {t1.cancelled()}")
    print(f"  Worker 3 cancelled? {t3.cancelled()}")

[08:20:01] 🟢 Worker 1 started (delay 2.0s)...
[08:20:01] 💣 Worker 2 started (will crash at 0.3s)...
[08:20:01] 🟢 Worker 3 started (delay 2.0s)...
[08:20:01] 💥 Worker 2 crashed!
[08:20:01] 🧹 Worker 1 received CancelledError! Cleaned up socket.
[08:20:01] 🧹 Worker 3 received CancelledError! Cleaned up socket.
🛑 Caught ExceptionGroup! All siblings were safely cancelled:
  Worker 1 cancelled? True
  Worker 3 cancelled? True


---
## 3. Handling Multiple Simultaneous Errors: `except*` Syntax

What if Task 1 raises `ValueError` AND Task 2 raises `TimeoutError` at the exact same time?
Traditional `try/except` could only catch ONE exception. 
Python 3.11 introduced **`except*`**, which lets you catch and handle multiple exception types independently out of a single `ExceptionGroup`!

In [2]:
import asyncio


async def bad_calc():
    await asyncio.sleep(0.1)
    raise ValueError("Invalid integer payload")

async def bad_lookup():
    await asyncio.sleep(0.1)
    raise KeyError("Missing user key in cache")

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(bad_calc())
        tg.create_task(bad_lookup())
except* KeyError as ke:
    print(f"Handling KeyErrors from TaskGroup: {[str(e) for e in ke.exceptions]}")
except* ValueError as ve:
    print(f"Handling ValueErrors from TaskGroup: {[str(e) for e in ve.exceptions]}")

print("✅ Both error types handled separately and cleanly!")

Handling KeyErrors from TaskGroup: ["'Missing user key in cache'"]
Handling ValueErrors from TaskGroup: ['Invalid integer payload']
✅ Both error types handled separately and cleanly!


---
## 4. Modern Timeouts: `asyncio.timeout` vs `asyncio.timeout_at`

- `async with asyncio.timeout(delay)`: Relative timeout (e.g. 1.5 seconds from now).
- `async with asyncio.timeout_at(loop.time() + deadline)`: Absolute deadline (ideal for end-to-end SLAs across multiple sequential operations).
- Unlike legacy `wait_for`, `asyncio.timeout` is cleaner and avoids creating extra wrapper tasks.

In [4]:
loop = asyncio.get_running_loop()
deadline = loop.time() + 0.5  # 500ms total budget for all 3 operations combined

try:
    async with asyncio.timeout_at(deadline):
        await asyncio.sleep(0.2)
        print("Step 1: DB query took 0.20s")
        await asyncio.sleep(0.2)
        print("Step 2: Cache check took 0.20s")
        # This 0.3s sleep exceeds the remaining 0.1s budget:
        await asyncio.sleep(0.3)
        print("Step 3 finished!")
except TimeoutError:
    print("Step 3: External API took too long and was aborted by timeout_at!")
    print("Remaining deadline expired.")

Step 1: DB query took 0.20s
Step 2: Cache check took 0.20s
Step 3: External API took too long and was aborted by timeout_at!
Remaining deadline expired.


---
## 5. Racing Concurrent Tasks: `asyncio.wait` with `FIRST_COMPLETED`

**Use Case (AI / Multi-Region Routing / LLM Fallbacks)**:
You want the fastest answer between 3 LLM providers (e.g., Anthropic, OpenAI, Local Ollama). 
Start all 3 at once, take the result from whoever answers first, and cancel the losers immediately!

In [5]:
async def query_llm_provider(provider: str, delay: float):
    await asyncio.sleep(delay)
    return f"{provider} reply ({delay}s)"

print("🏁 Racing OpenAI, Anthropic, and Groq for lowest latency...")
t_openai = asyncio.create_task(query_llm_provider("OpenAI", 0.6))
t_anthropic = asyncio.create_task(query_llm_provider("Anthropic", 0.4))
t_groq = asyncio.create_task(query_llm_provider("Groq", 0.15))

tasks = {t_openai, t_anthropic, t_groq}

# Wait only for FIRST_COMPLETED:
done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)

winner = done.pop()
print(f"🏆 Winner: {winner.result()}!")

# Cancel all slower pending tasks:
print(f"Cancelling {len(pending)} slower competitors...")
for p in pending:
    p.cancel()

await asyncio.gather(*pending, return_exceptions=True)
print("✅ Finished race cleanly without waiting for slow models.")

🏁 Racing OpenAI, Anthropic, and Groq for lowest latency...
🏆 Winner: Groq in 0.15s with response: 'Groq reply (0.15s)'!
Cancelling 2 slower competitors...
✅ Finished race cleanly without waiting for slow models.
